# 03 — DistilBERT Fine-Tune Sweep

**Goal:** characterise where a fine-tuned DistilBERT becomes competitive with Claude Sonnet 4.6 (macro-F1 = 0.8913 on Banking77 test, from notebook 02) — and at what cost.

## Two recipes, both run on the same data splits

- **v1_conservative** (already run): LR=2e-5, batch=32, EPOCHS_BY_N={50:20, ..., 5000:5}. Top result: macro-F1 ≈ 0.77 at n=5000. Doesn't cross Sonnet.
- **v2_retune** (this notebook will produce): LR=4e-5, batch=16, EPOCHS_BY_N revised up across the board. Diagnostic from v1: std=0.009 at n=5000 → model had converged at the low LR. Bumping LR + halving batch size doubles the per-epoch update count and pushes toward a better optimum.

Switch `RECIPE` in section 4 to re-run either version. Per-recipe outputs land in `results/finetune/{recipe}/` so the two experiments coexist for comparison in notebook 04.

## New training size: n=9000

Added to the sweep — uses 9,000 of the 9,203 available train-pool rows (after val + dev carve-out). Roughly matches the data regime of Casanueva 2020's published full-train results, lets us see the upper end of the data curve.

## Per-(n, seed) prediction files + softmax probabilities

Each run saves predictions AND the full 77-class softmax probabilities to `results/finetune/{recipe}/n{n}_seed{seed}.parquet`. Saving probs (not just hard labels) enables soft-vote ensembling across the 5 seeds in section 8.

## Sections

1. Environment setup (Colab clone / local no-op)
2. Imports + device detection
3. Load static data, pre-tokenize val + test
4. Recipe selection + hyperparameters
5. `train_one_run` (saves predictions + softmax probs)
6. Main sweep loop — 40 runs (8 sizes × 5 seeds), skip if exists
7. Aggregate per-(n, seed) parquets
8. Soft-vote ensemble across seeds — production-realistic headline metric
9. Per-(n, seed) and ensemble summary tables
10. Verify model checkpoint
11. (Colab only) zip + download results

## 1. Environment setup

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO_URL = 'https://github.com/louisgrochla/ds-llm-judge-vs-small-model.git'
    REPO_DIR = '/content/ds-llm-judge-vs-small-model'
    if not os.path.exists(REPO_DIR):
        get_ipython().system(f'git clone {REPO_URL} {REPO_DIR}')
    os.chdir(REPO_DIR)
    get_ipython().system("pip install -q 'datasets>=2.16,<4.0' transformers accelerate scikit-learn")
    print(f'Colab — working in {REPO_DIR}')
else:
    print('Local — using existing venv at .venv/')

print(f'Working dir: {os.getcwd()}')

## 2. Imports + device detection

In [ ]:
import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
    set_seed,
)
from sklearn.metrics import f1_score, accuracy_score

sys.path.insert(0, '.')
from src.data import (
    load_test_set,
    load_val_set,
    load_train_subset,
    load_label_names,
    TRAINING_SIZES,
    N_SEEDS,
)

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f'Device:           {device}')
print(f'Training sizes:   {TRAINING_SIZES}')
print(f'Seeds:            {N_SEEDS}')
print(f'Total runs:       {len(TRAINING_SIZES) * N_SEEDS}')

## 3. Load static data + pre-tokenize

Val and test are constant across all runs — tokenize them once.

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
MAX_LENGTH = 64
NUM_LABELS = 77

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

val_df = load_val_set()
test_df = load_test_set()
label_names = load_label_names()


def tokenize(df):
    enc = tokenizer(
        df['text'].tolist(),
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH,
        return_tensors='pt',
    )
    return {
        'input_ids': enc['input_ids'],
        'attention_mask': enc['attention_mask'],
        'labels': torch.tensor(df['label'].astype(int).tolist(), dtype=torch.long),
    }


class TokenizedDataset(Dataset):
    def __init__(self, encoded):
        self.encoded = encoded

    def __len__(self):
        return len(self.encoded['labels'])

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.encoded.items()}


val_encoded = tokenize(val_df)
test_encoded = tokenize(test_df)

print(f'Val:  {len(val_df):,} rows, {val_df["label"].nunique()} classes, tokenized')
print(f'Test: {len(test_df):,} rows, {test_df["label"].nunique()} classes, tokenized')

## 4. Recipe selection + hyperparameters

**Change `RECIPE` to switch between experiments.** Outputs are kept in separate folders so both can coexist for comparison in notebook 04.

### v1_conservative (already run on 2026-05-24)

Original conservative recipe — what most defaults-based fine-tuning code would produce. Reached macro-F1 ≈ 0.77 at n=5000 with std=0.009. The low std confirmed the model had converged at this LR — more epochs alone wouldn't help.

### v2_retune (proposed)

**Diagnosis:** v1's classifier head wasn't getting enough update magnitude at LR=2e-5. The 60K randomly-initialised classifier weights need more aggressive moves than the pre-trained 66M body weights would tolerate at high LR.

**Intervention:**
- LR=4e-5 — moderately higher than v1, still well below published-baseline aggression (Casanueva 2020 used 5e-5). Tests whether a midpoint LR is sufficient.
- batch=16 (down from 32) — doubles gradient updates per epoch at the same epoch count.
- epochs scaled up everywhere — gives the higher LR more steps to find a better optimum.

Total compute on T4 still under 90 minutes for the full 40-run sweep (8 sizes × 5 seeds).

Justification for the writeup: "I observed v1 converged at low LR (std=0.009 at n=5000), bumped LR moderately + reduced batch size to increase update density, kept epochs constrained to stay within Colab T4 budget."

In [ ]:
RECIPE = 'v2_retune'   # change to 'v1_conservative' to reproduce the initial sweep

RECIPES = {
    'v1_conservative': {
        'LR': 2e-5,
        'BATCH_SIZE': 32,
        'EPOCHS_BY_N': {50: 20, 100: 15, 250: 10, 500: 8, 1000: 6, 2500: 5, 5000: 5, 9000: 5},
    },
    'v2_retune': {
        'LR': 4e-5,
        'BATCH_SIZE': 16,
        'EPOCHS_BY_N': {50: 30, 100: 25, 250: 20, 500: 15, 1000: 12, 2500: 10, 5000: 8, 9000: 8},
    },
}

HP = RECIPES[RECIPE]
LR = HP['LR']
BATCH_SIZE = HP['BATCH_SIZE']
EPOCHS_BY_N = HP['EPOCHS_BY_N']
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.1
EARLY_STOPPING_PATIENCE = 3   # bumped from 2 — at higher LR, val loss can wobble before improving

FINETUNE_DIR = Path('results/finetune') / RECIPE
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = Path(f'results/checkpoints/{RECIPE}_n9000_seed0')


def epochs_for_n(n):
    return EPOCHS_BY_N.get(n, 5)


print(f'Recipe:           {RECIPE}')
print(f'Learning rate:    {LR}')
print(f'Batch size:       {BATCH_SIZE}')
print(f'Epochs by n:      {EPOCHS_BY_N}')
print(f'Weight decay:     {WEIGHT_DECAY}')
print(f'Warmup fraction:  {WARMUP_FRAC}')
print(f'Early stop after: {EARLY_STOPPING_PATIENCE} bad val epochs')
print(f'Output dir:       {FINETUNE_DIR}/')
print(f'Checkpoint dir:   {CKPT_PATH}/  (saved for n=max(N), seed=0)')

## 5. `train_one_run` — saves predictions AND softmax probabilities

Same fine-tuning loop as before (AdamW + linear warmup + early stopping on val loss). New: saves the full 77-class softmax distribution per test row, enabling soft-vote ensembling in section 8.

In [ ]:
def train_one_run(n: int, seed: int, save_model_path: Path | None = None) -> dict:
    set_seed(seed)
    epochs = epochs_for_n(n)

    train_df = load_train_subset(n, seed)
    train_encoded = tokenize(train_df)

    train_loader = DataLoader(TokenizedDataset(train_encoded), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(TokenizedDataset(val_encoded), batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(TokenizedDataset(test_encoded), batch_size=BATCH_SIZE, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = max(1, len(train_loader) * epochs)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(WARMUP_FRAC * total_steps),
        num_training_steps=total_steps,
    )

    best_val_loss = float('inf')
    best_state = None
    patience_left = EARLY_STOPPING_PATIENCE
    epochs_run = 0

    start_time = time.time()

    for epoch in range(epochs):
        epochs_run += 1
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                val_losses.append(model(**batch).loss.item())
        val_loss = float(np.mean(val_losses))

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone().detach().cpu() for k, v in model.state_dict().items()}
            patience_left = EARLY_STOPPING_PATIENCE
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # Test predictions + softmax probs
    model.eval()
    all_preds = []
    all_probs = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(**batch).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
            all_preds.extend(logits.argmax(dim=-1).cpu().tolist())
    all_probs = np.concatenate(all_probs, axis=0)  # (n_test, 77)

    train_time_s = time.time() - start_time

    preds_df = test_df[['text', 'label']].copy()
    preds_df = preds_df.rename(columns={'label': 'true_label'})
    preds_df['pred_label'] = all_preds
    preds_df['true_intent'] = preds_df['true_label'].map(lambda i: label_names[int(i)])
    preds_df['pred_intent'] = preds_df['pred_label'].map(lambda i: label_names[int(i)])
    preds_df['is_correct'] = preds_df['true_label'] == preds_df['pred_label']
    preds_df['training_size'] = n
    preds_df['seed'] = seed
    preds_df['probs'] = list(all_probs.astype(np.float32))   # list of 77-element arrays

    out_path = FINETUNE_DIR / f'n{n}_seed{seed}.parquet'
    preds_df.to_parquet(out_path)

    metadata = {
        'recipe': RECIPE,
        'training_size': n,
        'seed': seed,
        'epochs_planned': epochs,
        'epochs_run': epochs_run,
        'train_time_s': round(train_time_s, 1),
        'best_val_loss': round(best_val_loss, 4),
        'test_accuracy': float(preds_df['is_correct'].mean()),
        'test_macro_f1': float(f1_score(preds_df['true_label'], preds_df['pred_label'], average='macro', zero_division=0)),
    }

    if save_model_path is not None:
        save_model_path.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(save_model_path)
        tokenizer.save_pretrained(save_model_path)

    del model, optimizer, scheduler
    if device == 'cuda':
        torch.cuda.empty_cache()

    return metadata


print('train_one_run defined.')

## 6. Main sweep loop

Iterates over (n, seed) pairs. Skips any combination whose prediction file already exists, so Colab disconnects don't lose completed runs. Saves the checkpoint for n=max(N), seed=0 — that's the model we ship to HF Hub and serve in the Gradio demo.

In [ ]:
MAX_N = max(TRAINING_SIZES)
all_metadata = []
completed = 0
skipped = 0

for n in TRAINING_SIZES:
    for seed in range(N_SEEDS):
        preds_path = FINETUNE_DIR / f'n{n}_seed{seed}.parquet'
        is_checkpoint_run = (n == MAX_N and seed == 0)
        save_path = CKPT_PATH if is_checkpoint_run else None

        if preds_path.exists():
            if is_checkpoint_run and not CKPT_PATH.exists():
                print(f'  n={n:<5} seed={seed}: predictions exist but no checkpoint — retraining to save model')
                meta = train_one_run(n, seed, save_model_path=save_path)
                completed += 1
                print(f'    done in {meta["train_time_s"]:.1f}s — test macro-F1 = {meta["test_macro_f1"]:.4f}')
                all_metadata.append(meta)
            else:
                skipped += 1
                print(f'  n={n:<5} seed={seed}: already done, skipping')
            continue

        print(f'  n={n:<5} seed={seed}: training ({epochs_for_n(n)} epochs)...', end='', flush=True)
        meta = train_one_run(n, seed, save_model_path=save_path)
        completed += 1
        print(f' done in {meta["train_time_s"]:.1f}s — macro-F1 = {meta["test_macro_f1"]:.4f}')
        all_metadata.append(meta)

print(f'\n{completed} runs completed, {skipped} skipped.')

if all_metadata:
    metadata_df = pd.DataFrame(all_metadata)
    metadata_path = FINETUNE_DIR / 'run_metadata.parquet'
    if metadata_path.exists():
        existing = pd.read_parquet(metadata_path)
        metadata_df = pd.concat([existing, metadata_df], ignore_index=True).drop_duplicates(
            subset=['training_size', 'seed'], keep='last'
        )
    metadata_df.to_parquet(metadata_path)
    print(f'Saved {metadata_path}')

## 7. Aggregate per-(n, seed) parquets

Concatenates all 40 prediction files into one `results/finetune_predictions_{recipe}.parquet`. Used by notebook 04 for the bootstrap test.

In [ ]:
all_files = sorted(FINETUNE_DIR.glob('n*_seed*.parquet'))
if not all_files:
    print('No prediction files yet — run section 6 first.')
else:
    dfs = [pd.read_parquet(p) for p in all_files]
    combined = pd.concat(dfs, ignore_index=True)
    combined_path = Path(f'results/finetune_predictions_{RECIPE}.parquet')
    combined.to_parquet(combined_path)
    runs = len(all_files)
    print(f'Aggregated {runs} runs × {len(test_df):,} test predictions = {len(combined):,} rows')
    print(f'Saved {combined_path}')

## 8. Soft-vote ensemble across seeds

For each training size, average the 5 seeds' softmax probability distributions per test example, then argmax. This is what a production deployment would do — single-seed runs vary substantially (especially at small n), and ensembling is the standard mitigation.

Why this matters for the writeup: the honest comparison to Sonnet 4.6 isn't "the best single seed of our fine-tune" (cherry-picking) — it's "what a production deployment would actually serve." That's the ensemble.

In [ ]:
if not all_files:
    print('No predictions to ensemble yet.')
else:
    ensemble_rows = []
    for n in TRAINING_SIZES:
        seed_files = sorted(FINETUNE_DIR.glob(f'n{n}_seed*.parquet'))
        if len(seed_files) < 2:
            continue

        # Stack probs across seeds: shape (n_seeds, n_test, 77)
        seed_probs = np.stack([
            np.array(pd.read_parquet(p)['probs'].tolist()) for p in seed_files
        ])
        mean_probs = seed_probs.mean(axis=0)               # (n_test, 77)
        ensemble_preds = mean_probs.argmax(axis=-1)        # (n_test,)

        y_true = test_df['label'].values
        ens_acc = float(accuracy_score(y_true, ensemble_preds))
        ens_f1 = float(f1_score(y_true, ensemble_preds, average='macro', zero_division=0))

        ensemble_rows.append({
            'training_size': n,
            'n_seeds': len(seed_files),
            'ensemble_accuracy': ens_acc,
            'ensemble_macro_f1': ens_f1,
        })

    ensemble_df = pd.DataFrame(ensemble_rows)
    ensemble_path = Path(f'results/finetune_ensemble_{RECIPE}.parquet')
    ensemble_df.to_parquet(ensemble_path)
    print(f'Saved {ensemble_path}')
    print()
    print('Ensemble (5-seed soft-vote) macro-F1 per n:')
    print(ensemble_df.to_string(index=False))

## 9. Single-seed + ensemble summary

Shows the per-seed variance AND the ensemble headline — what notebook 04 will use for its crossover plot.

In [ ]:
if not all_files:
    print('No runs to summarise.')
else:
    summary_rows = []
    for p in all_files:
        df = pd.read_parquet(p)
        n = int(df['training_size'].iloc[0])
        seed = int(df['seed'].iloc[0])
        summary_rows.append({
            'training_size': n,
            'seed': seed,
            'macro_f1': float(f1_score(df['true_label'], df['pred_label'], average='macro', zero_division=0)),
        })
    summary = pd.DataFrame(summary_rows)

    print('Single-seed macro-F1 mean ± std at each n:')
    agg = summary.groupby('training_size')['macro_f1'].agg(['mean', 'std']).round(4)
    print(agg.to_string())
    print()
    print('Ensemble (soft-vote across 5 seeds) — production-realistic:')
    print(ensemble_df[['training_size', 'ensemble_macro_f1']].round(4).to_string(index=False))
    print()
    print(f'Sonnet 4.6 baseline (notebook 02): 0.8913 macro-F1 on test')

## 10. Verify model checkpoint

In [ ]:
if CKPT_PATH.exists():
    files_in_ckpt = sorted(p.name for p in CKPT_PATH.iterdir())
    size_mb = sum(p.stat().st_size for p in CKPT_PATH.iterdir()) / (1024 * 1024)
    print(f'Checkpoint at {CKPT_PATH}/ — {size_mb:.1f} MB')
    print(f'Files: {files_in_ckpt}')
else:
    print(f'No checkpoint at {CKPT_PATH}/ — re-run section 6 for n={MAX_N}, seed=0')

## 11. (Colab only) Zip + download results

In [ ]:
if not IN_COLAB:
    print('Local — results already in your working directory; skip the download step.')
else:
    import shutil

    ZIP_BASE = '/content/finetune_results'
    if os.path.exists(ZIP_BASE):
        shutil.rmtree(ZIP_BASE)
    os.makedirs(f'{ZIP_BASE}/results/finetune/{RECIPE}', exist_ok=True)
    os.makedirs(f'{ZIP_BASE}/results/checkpoints', exist_ok=True)

    for p in FINETUNE_DIR.glob('*.parquet'):
        shutil.copy(p, f'{ZIP_BASE}/results/finetune/{RECIPE}/')
    for src_name in [f'finetune_predictions_{RECIPE}.parquet', f'finetune_ensemble_{RECIPE}.parquet']:
        src = Path('results') / src_name
        if src.exists():
            shutil.copy(src, f'{ZIP_BASE}/results/')
    if CKPT_PATH.exists():
        shutil.copytree(CKPT_PATH, f'{ZIP_BASE}/results/checkpoints/{CKPT_PATH.name}')

    shutil.make_archive('/content/finetune_results', 'zip', ZIP_BASE)
    from google.colab import files as colab_files
    colab_files.download('/content/finetune_results.zip')
    print('Downloaded — unzip into the repo root on your Mac.')